In [5]:
from rdflib import Graph
g = Graph(); g.parse("ORDO_en_4.7.owl", format="xml")
print("Triples:", len(g))             # should be large (millions), not single digits

# Count Orphanet_* classes
q = """
SELECT (COUNT(DISTINCT ?d) AS ?n) WHERE {
  ?d a <http://www.w3.org/2002/07/owl#Class> .
  FILTER regex(STR(?d), "Orphanet_\\\\d+$")
}
"""
print("Orphanet_* classes:", list(g.query(q))[0][0])


Triples: 587954
Orphanet_* classes: 15772
Orphanet_* classes: 15772


In [6]:
# pip install rdflib pandas
import re
import pandas as pd
from rdflib import Graph

# --- point this at your full file ---
ORDO_PATH = "ORDO_en_4.7.owl"

def iri_to_orpha(iri: str) -> str | None:
    m = re.search(r"Orphanet_(\d+)$", iri)
    return m.group(1) if m else None

def get_rare_diseases_basic(ordo_path: str) -> pd.DataFrame:
    g = Graph(); g.parse(ordo_path, format="xml")

    q = """
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    PREFIX owl:  <http://www.w3.org/2002/07/owl#>
    PREFIX oboInOwl: <http://www.geneontology.org/formats/oboInOwl#>
    PREFIX dcterms: <http://purl.org/dc/terms/>
    PREFIX ebi:  <http://www.ebi.ac.uk/efo/>

    SELECT DISTINCT ?d (COALESCE(?l1, ?l2) AS ?name) ?alias ?xref WHERE {
      # treat any OWL class whose IRI ends with Orphanet_<digits> as a disease
      ?d a owl:Class .
      FILTER regex(STR(?d), "Orphanet_\\\\d+$")

      # canonical name: prefer rdfs:label, else skos:prefLabel; accept en or unlabeled
      OPTIONAL { ?d rdfs:label ?l1 }
      OPTIONAL { ?d skos:prefLabel ?l2 }
      BIND(COALESCE(?l1, ?l2) AS ?nm)
      FILTER(BOUND(?nm))
      FILTER( lang(?nm) = "" || langMatches(lang(?nm), "en") )
      BIND(?nm AS ?name)

      # aliases / other names (collect many common properties)
      OPTIONAL { ?d skos:altLabel ?alias }
      OPTIONAL { ?d oboInOwl:hasExactSynonym ?alias }
      OPTIONAL { ?d oboInOwl:hasRelatedSynonym ?alias }
      OPTIONAL { ?d dcterms:alternative ?alias }
      OPTIONAL { ?d ebi:alternative_term ?alias }

      # external mappings / xrefs
      OPTIONAL { ?d skos:exactMatch ?xref }
      OPTIONAL { ?d skos:closeMatch ?xref }
      OPTIONAL { ?d owl:sameAs ?xref }
      OPTIONAL { ?d rdfs:seeAlso ?xref }
      OPTIONAL { ?d oboInOwl:hasDbXref ?xref }
    }
    """
    rows = [{k: (str(v) if v is not None else None) for k, v in r.asdict().items()} for r in g.query(q)]

    # fold rows -> one per disease, deduping aliases/xrefs
    bucket = {}
    for r in rows:
        d = r["d"]; name = r["name"]; alias = r.get("alias"); xref = r.get("xref")
        rec = bucket.setdefault(d, {"uri": d, "name": name, "aliases": set(), "xrefs": set()})
        if alias and alias != name:
            # keep only english or unlabeled aliases (mirrors main label rule)
            if not alias or "@en" in alias or "@" not in alias:
                rec["aliases"].add(alias if "@" not in alias else alias.rsplit("@",1)[0])
        if xref:
            rec["xrefs"].add(xref)

    df = pd.DataFrame([{
        "uri": v["uri"],
        "orpha": iri_to_orpha(v["uri"]),
        "name": v["name"],
        "aliases": sorted(v["aliases"]),
        "xrefs": sorted(v["xrefs"])
    } for v in bucket.values()])

    # drop the 6 umbrella/bucket labels you saw earlier
    drop_labels = {
        "Disease",
        "Malformation syndrome",
        "Biological anomaly",
        "Morphological anomaly",
        "Clinical syndrome",
        "Particular clinical situation in a disease or syndrome",
    }
    df = df[~df["name"].isin(drop_labels)].reset_index(drop=True)
    return df

df = get_rare_diseases_basic(ORDO_PATH)
print("Rare diseases (approx):", len(df))
print(df.head(15))

# Save to CSV for inspection
df.to_csv("ordo_rare_diseases_basic.csv", index=False)
print("Wrote: ordo_rare_diseases_basic.csv")


Rare diseases (approx): 15766
                                          uri   orpha  \
0       http://www.orpha.net/ORDO/Orphanet_10      10   
1      http://www.orpha.net/ORDO/Orphanet_100     100   
2     http://www.orpha.net/ORDO/Orphanet_1000    1000   
3   http://www.orpha.net/ORDO/Orphanet_100000  100000   
4   http://www.orpha.net/ORDO/Orphanet_100001  100001   
5   http://www.orpha.net/ORDO/Orphanet_100002  100002   
6   http://www.orpha.net/ORDO/Orphanet_100003  100003   
7   http://www.orpha.net/ORDO/Orphanet_100006  100006   
8   http://www.orpha.net/ORDO/Orphanet_100008  100008   
9   http://www.orpha.net/ORDO/Orphanet_100011  100011   
10  http://www.orpha.net/ORDO/Orphanet_100012  100012   
11  http://www.orpha.net/ORDO/Orphanet_100013  100013   
12  http://www.orpha.net/ORDO/Orphanet_100014  100014   
13  http://www.orpha.net/ORDO/Orphanet_100015  100015   
14  http://www.orpha.net/ORDO/Orphanet_100016  100016   

                                                 name  \


In [3]:
# pip install rdflib pandas
import re, pandas as pd
from rdflib import Graph

ORDO = "/home/guests/andreea_magureanu/projects/rare_disease_pipeline/paper_filter/ordo_orphanet.owl"  # your local file (RDF/XML)

g = Graph()
g.parse(ORDO, format="xml")

q = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX owl:  <http://www.w3.org/2002/07/owl#>
PREFIX oboInOwl: <http://www.geneontology.org/formats/oboInOwl#>
PREFIX ebi:  <http://www.ebi.ac.uk/efo/>
SELECT DISTINCT ?d ?label ?alias ?xref WHERE {
  ?d rdfs:label ?label .
  FILTER(langMatches(lang(?label), "en"))
  FILTER regex(STR(?d), "Orphanet_\\\\d+$")  # treat Orphanet_* IRIs as diseases

  OPTIONAL { ?d skos:altLabel ?alias FILTER(langMatches(lang(?alias), "en")) }
  OPTIONAL { ?d oboInOwl:hasExactSynonym ?alias }
  OPTIONAL { ?d oboInOwl:hasRelatedSynonym ?alias }
  OPTIONAL { ?d ebi:alternative_term ?alias }

  OPTIONAL { ?d skos:exactMatch ?xref }
  OPTIONAL { ?d skos:closeMatch ?xref }
  OPTIONAL { ?d owl:sameAs ?xref }
  OPTIONAL { ?d rdfs:seeAlso ?xref }
  OPTIONAL { ?d oboInOwl:hasDbXref ?xref }
}
LIMIT 500
"""
rows = [{k: (str(v) if v else None) for k,v in r.asdict().items()} for r in g.query(q)]

# fold aliases/xrefs
bucket = {}
for r in rows:
    d = r["d"]; lab = r["label"]; alias=r.get("alias"); xref=r.get("xref")
    rec = bucket.setdefault(d, {"uri": d, "label": lab, "aliases": set(), "xrefs": set()})
    if alias and alias != lab: rec["aliases"].add(alias)
    if xref: rec["xrefs"].add(xref)

df = pd.DataFrame([{"uri":v["uri"], "label":v["label"],
                    "aliases":sorted(v["aliases"]), "xrefs":sorted(v["xrefs"])}
                   for v in bucket.values()])
print(df.head(10))


                                         uri  \
0  http://www.orpha.net/ORDO/Orphanet_377788   
1  http://www.orpha.net/ORDO/Orphanet_377789   
2  http://www.orpha.net/ORDO/Orphanet_377790   
3  http://www.orpha.net/ORDO/Orphanet_377791   
4  http://www.orpha.net/ORDO/Orphanet_377792   
5  http://www.orpha.net/ORDO/Orphanet_377793   

                                               label aliases xrefs  
0                                            Disease      []    []  
1                              Malformation syndrome      []    []  
2                                 Biological anomaly      []    []  
3                              Morphological anomaly      []    []  
4                                  Clinical syndrome      []    []  
5  Particular clinical situation in a disease or ...      []    []  


In [4]:
print(df.shape)

(6, 4)


In [ ]:

import re
from typing import Dict, List, Optional, Iterable
import pandas as pd
from rdflib import Graph, Namespace
from neo4j import GraphDatabase, basic_auth


# ===================== CONFIG =====================

ORDO_PATH = "/home/guests/andreea_magureanu/projects/rare_disease_pipeline/paper_filter/ordo_orphanet.owl"   # path to ORDO RDF/XML file

NEO4J_URI  = "neo4j+s://.databases.neo4j.io"
NEO4J_USER = ""
NEO4J_PASS = ""

BATCH = 1000  # UNWIND batch size

# Toggle optional extras (kept TRUE since you said "if there are more relevant entities")
INCLUDE_INHERITANCE = True
INCLUDE_AGE_OF_ONSET = True


# ===================== HELPERS =====================

def load_graph(path: str) -> Graph:
    g = Graph()
    g.parse(path, format="xml")
    return g

def iri_to_orpha(iri: str) -> Optional[str]:
    m = re.search(r"Orphanet_(\d+)$", iri)
    return m.group(1) if m else None

def chunks(it: Iterable[dict], n: int):
    buf = []
    for row in it:
        buf.append(row)
        if len(buf) >= n:
            yield buf
            buf = []
    if buf:
        yield buf

def find_root_by_label(g: Graph, pattern: str) -> Optional[str]:
    q = f"""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT DISTINCT ?c ?label WHERE {{
      ?c rdfs:label ?label .
      FILTER(langMatches(lang(?label), "en"))
      FILTER(regex(?label, "{pattern}", "i"))
    }}
    """
    rows = list(g.query(q))
    if not rows:
        return None

    best = None
    best_n = -1
    for r in rows:
        c = str(r["c"])
        qc = f"SELECT (COUNT(?x) AS ?n) WHERE {{ ?x <http://www.w3.org/2000/01/rdf-schema#subClassOf>+ <{c}> . }}"
        n = list(g.query(qc))[0][0].toPython()
        if n > best_n:
            best, best_n = c, n
    return best

def find_inactive_root(g: Graph) -> Optional[str]:
    return find_root_by_label(g, r"^inactive clinical entity$|^not rare in europe$|^obsolete$")



def extract_diseases(g: Graph, gene_root: Optional[str], inactive_root: Optional[str]) -> pd.DataFrame:
    """
    Pull 'diseases' as any ORDO class whose IRI ends with Orphanet_<digits>,
    excluding classes under Genetic material and Inactive branches.
    """
    q = f"""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl:  <http://www.w3.org/2002/07/owl#>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

    SELECT DISTINCT ?d ?name ?alias ?xref WHERE {{
      ?d rdfs:label ?name .
      FILTER(langMatches(lang(?name), "en"))
      FILTER regex(STR(?d), "Orphanet_\\\\d+$")  # keep only disease-like IRIs

      {f"FILTER NOT EXISTS {{ ?d rdfs:subClassOf+ <{gene_root}> . }}" if gene_root else ""}
      {f"FILTER NOT EXISTS {{ ?d rdfs:subClassOf+ <{inactive_root}> . }}" if inactive_root else ""}

      OPTIONAL {{ ?d skos:altLabel ?alias FILTER(langMatches(lang(?alias), "en")) }}
      OPTIONAL {{ ?d <http://www.geneontology.org/formats/oboInOwl#hasExactSynonym> ?alias }}
      OPTIONAL {{ ?d <http://www.geneontology.org/formats/oboInOwl#hasRelatedSynonym> ?alias }}
      OPTIONAL {{ ?d <http://purl.org/dc/terms/alternative> ?alias }}
      OPTIONAL {{ ?d <http://www.ebi.ac.uk/efo/alternative_term> ?alias }}

      OPTIONAL {{ ?d skos:exactMatch ?xref }}
      OPTIONAL {{ ?d skos:closeMatch ?xref }}
      OPTIONAL {{ ?d owl:sameAs ?xref }}
      OPTIONAL {{ ?d rdfs:seeAlso ?xref }}
      OPTIONAL {{ ?d <http://www.geneontology.org/formats/oboInOwl#hasDbXref> ?xref }}
    }}
    """
    raw = [{k: (str(v) if v else None) for k, v in r.asdict().items()} for r in g.query(q)]
    bucket: Dict[str, dict] = {}
    for r in raw:
        uri, name, alias, xref = r["d"], r["name"], r.get("alias"), r.get("xref")
        rec = bucket.setdefault(uri, {"uri": uri, "name": name, "orpha": iri_to_orpha(uri),
                                      "aliases": set(), "xrefs": set()})
        if alias and alias != name: rec["aliases"].add(alias)
        if xref: rec["xrefs"].add(xref)
    return pd.DataFrame([{
        "uri": rec["uri"], "name": rec["name"], "orpha": rec["orpha"],
        "aliases": sorted(rec["aliases"]), "xrefs": sorted(rec["xrefs"]),
    } for rec in bucket.values()])



def extract_causal_gene_edges(g: Graph, gene_root: Optional[str]) -> pd.DataFrame:
    """
    Disease (IRI ending Orphanet_<digits>) ←causal– Gene (under Genetic material).
    """
    if not gene_root:
        return pd.DataFrame()

    q = f"""
    PREFIX owl:  <http://www.w3.org/2002/07/owl#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT DISTINCT ?d ?g ?dName ?gName ?rel ?relLabel WHERE {{
      ?d rdfs:label ?dName .
      FILTER(langMatches(lang(?dName), "en"))
      FILTER regex(STR(?d), "Orphanet_\\\\d+$")

      ?g rdfs:subClassOf+ <{gene_root}> ;
         rdfs:label ?gName .
      FILTER(langMatches(lang(?gName), "en"))

      {{
        ?d rdfs:subClassOf ?r1 .
        ?r1 a owl:Restriction ; owl:onProperty ?rel ; owl:someValuesFrom ?g .
      }} UNION {{
        ?g rdfs:subClassOf ?r2 .
        ?r2 a owl:Restriction ; owl:onProperty ?rel ; owl:someValuesFrom ?d .
      }}

      OPTIONAL {{ ?rel rdfs:label ?relLabel FILTER(langMatches(lang(?relLabel), "en")) }}
      FILTER regex(COALESCE(?relLabel, STR(?rel)), "(?i)(disease-causing|causal)")
    }}
    """
    rows = [{k: (str(v) if v else None) for k, v in row.asdict().items()} for row in g.query(q)]
    seen = set(); out = []
    for r in rows:
        key = (r["d"], r["g"])
        if key in seen: continue
        seen.add(key)
        out.append({
            "disease_uri": r["d"], "gene_uri": r["g"],
            "disease": r["dName"], "gene": r["gName"],
            "rel": r.get("relLabel") or "disease-causing mutation in"
        })
    return pd.DataFrame(out)





def extract_one_to_object(g: Graph, disease_root: str, object_root: str, label_regex: str) -> pd.DataFrame:
    """
    Generic extractor for Disease -> (Object under object_root) via relation whose label
    matches label_regex (e.g., inheritance, age of onset).
    """
    q = f"""
    PREFIX owl:  <http://www.w3.org/2002/07/owl#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT DISTINCT ?d ?o ?dName ?oName ?rel ?relLabel WHERE {{
      ?d rdfs:subClassOf+ <{disease_root}> .
      ?d rdfs:label ?dName .
      FILTER(langMatches(lang(?dName), "en"))

      {{
        ?d rdfs:subClassOf ?r .
        ?r a owl:Restriction ; owl:onProperty ?rel ; owl:someValuesFrom ?o .
      }} UNION {{
        ?o rdfs:subClassOf ?r2 .
        ?r2 a owl:Restriction ; owl:onProperty ?rel ; owl:someValuesFrom ?d .
      }}

      ?o rdfs:subClassOf+ <{object_root}> .
      ?o rdfs:label ?oName .
      FILTER(langMatches(lang(?oName), "en"))

      OPTIONAL {{ ?rel rdfs:label ?relLabel FILTER(langMatches(lang(?relLabel), "en")) }}
      FILTER regex( COALESCE(?relLabel, STR(?rel)), "{label_regex}" )
    }}
    """
    rows = [ {k: (str(v) if v else None) for k, v in r.asdict().items()} for r in g.query(q) ]
    seen = set()
    out = []
    for r in rows:
        key = (r["d"], r["o"])
        if key in seen:
            continue
        seen.add(key)
        out.append({
            "disease_uri": r["d"], "object_uri": r["o"],
            "disease": r["dName"], "object": r["oName"],
        })
    return pd.DataFrame(out)


# ===================== NEO4J LOAD =====================

def neo4j_driver():
    return GraphDatabase.driver(NEO4J_URI, auth=basic_auth(NEO4J_USER, NEO4J_PASS))

def ensure_constraints(driver):
    cypher = """
    CREATE CONSTRAINT IF NOT EXISTS FOR (d:Disease)   REQUIRE d.uri IS UNIQUE;
    CREATE CONSTRAINT IF NOT EXISTS FOR (g:Gene)      REQUIRE g.uri IS UNIQUE;
    CREATE CONSTRAINT IF NOT EXISTS FOR (p:Phenotype) REQUIRE p.uri IS UNIQUE;
    # CREATE CONSTRAINT IF NOT EXISTS FOR (i:Inheritance) REQUIRE i.uri IS UNIQUE;
    # CREATE CONSTRAINT IF NOT EXISTS FOR (a:AgeOfOnset) REQUIRE a.uri IS UNIQUE;
    """
    with driver.session() as s:
        for stmt in [x.strip() for x in cypher.strip().split(";") if x.strip()]:
            s.run(stmt)

def load_diseases(driver, df: pd.DataFrame):
   
    q = """
    UNWIND $rows AS row
    MERGE (d:Disease {uri: row.uri})
      ON CREATE SET d.name = row.name,
                    d.orpha = row.orpha,
                    d.aliases = row.aliases,
                    d.xrefs = row.xrefs
      ON MATCH SET  d.name = coalesce(row.name, d.name),
                    d.orpha = coalesce(row.orpha, d.orpha),
                    d.aliases = row.aliases,
                    d.xrefs = row.xrefs
    """
    rows = df.to_dict(orient="records")
    with driver.session() as s:
        for batch in chunks(rows, BATCH):
            s.run(q, rows=batch)

def load_gene_edges(driver, df: pd.DataFrame):
    q = """
    UNWIND $rows AS row
    MERGE (g:Gene {uri: row.gene_uri})
      ON CREATE SET g.name = row.gene
      ON MATCH SET  g.name = coalesce(row.gene, g.name)
    MERGE (d:Disease {uri: row.disease_uri})
    MERGE (d)-[r:CAUSED_BY]->(g)
      ON CREATE SET r.source = "ORDO", r.label = row.rel
    """
    rows = df.to_dict(orient="records")
    with driver.session() as s:
        for batch in chunks(rows, BATCH):
            s.run(q, rows=batch)



def load_inheritance_edges(driver, df: pd.DataFrame):
    q = """
    UNWIND $rows AS row
    MERGE (i:Inheritance {uri: row.object_uri})
      ON CREATE SET i.name = row.object
      ON MATCH SET  i.name = coalesce(row.object, i.name)
    MERGE (d:Disease {uri: row.disease_uri})
    MERGE (d)-[:HAS_INHERITANCE]->(i)
    """
    rows = df.to_dict(orient="records")
    with driver.session() as s:
        for batch in chunks(rows, BATCH):
            s.run(q, rows=batch)

def load_age_edges(driver, df: pd.DataFrame):
    q = """
    UNWIND $rows AS row
    MERGE (a:AgeOfOnset {uri: row.object_uri})
      ON CREATE SET a.name = row.object
      ON MATCH SET  a.name = coalesce(row.object, a.name)
    MERGE (d:Disease {uri: row.disease_uri})
    MERGE (d)-[:HAS_AGE_OF_ONSET]->(a)
    """
    rows = df.to_dict(orient="records")
    with driver.session() as s:
        for batch in chunks(rows, BATCH):
            s.run(q, rows=batch)


# ===================== MAIN =====================

def main():
    print("Loading ORDO…")
g = load_graph(ORDO_PATH)

print("Locating ontology branches by label…")
gene_root    = find_root_by_label(g, r"^genetic material$")
inactive_root = find_inactive_root(g)
print(f" gene_root:    {gene_root or '—'}")
print(f" inactive_root:{inactive_root or '—'}")

print("\nExtracting diseases (names, aliases, xrefs, ORPHAcode)…")
df_dis = extract_diseases(g, gene_root, inactive_root)
print(f"  Diseases: {len(df_dis)}")

print("Extracting causal Disease–Gene relations…")
df_gene = extract_causal_gene_edges(g, gene_root)
print(f"  Causal edges: {len(df_gene)}")

print("\nLoading into Neo4j…")

if __name__ == "__main__":
    main()


Locating ontology branches by label…
 gene_root:    —
 inactive_root:—

Extracting diseases (names, aliases, xrefs, ORPHAcode)…
 gene_root:    —
 inactive_root:—

Extracting diseases (names, aliases, xrefs, ORPHAcode)…
  Diseases: 6
Extracting causal Disease–Gene relations…
  Causal edges: 0

Loading into Neo4j…
  Diseases: 6
Extracting causal Disease–Gene relations…
  Causal edges: 0

Loading into Neo4j…


CypherSyntaxError: {code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input '#': expected 'ALTER', 'ORDER BY', 'CALL', 'USING PERIODIC COMMIT', 'CREATE', 'LOAD CSV', 'START DATABASE', 'STOP DATABASE', 'DEALLOCATE', 'DELETE', 'DENY', 'DETACH', 'DROP', 'DRYRUN', 'FINISH', 'FOREACH', 'GRANT', 'INSERT', 'LIMIT', 'MATCH', 'MERGE', 'NODETACH', 'OFFSET', 'OPTIONAL', 'REALLOCATE', 'REMOVE', 'RENAME', 'RETURN', 'REVOKE', 'ENABLE SERVER', 'SET', 'SHOW', 'SKIP', 'TERMINATE', 'UNWIND', 'USE' or 'WITH' (line 1, column 1 (offset: 0))
"# CREATE CONSTRAINT IF NOT EXISTS FOR (i:Inheritance) REQUIRE i.uri IS UNIQUE"
 ^}

In [ ]:
from neo4j import GraphDatabase
from typing import Optional, List, Dict, Union
import logging
from contextlib import contextmanager

class Neo4jConnection:
    """A class to manage Neo4j database connections for biomedical data"""
    
    VALID_TYPES = {'disease', 'gene', 'genotype', 'phenotype', 'treatment'}
    
    def __init__(self, uri: str, auth: tuple, database: str):
        self.uri = uri
        self.auth = auth
        self.database = database
        self.driver = None
        
    def connect(self) -> None:
        """Creates a connection to the Neo4j database"""
        try:
            self.driver = GraphDatabase.driver(self.uri, auth=self.auth)
            self.driver.verify_connectivity()
            logging.info("Connected to Neo4j database")
        except Exception as e:
            logging.error(f"Failed to connect to Neo4j: {str(e)}")
            raise
            
    @contextmanager
    def get_session(self):
        """Context manager for database sessions"""
        if not self.driver:
            self.connect()
            
        session = None
        try:
            session = self.driver.session(database=self.database)
            yield session
        finally:
            if session:
                session.close()
                
    def close(self):
        """Closes the database connection"""
        if self.driver:
            self.driver.close()
            self.driver = None
            logging.info("Neo4j connection closed")

    def create_biomedical_node(self, 
                             canonical_name: str,
                             node_type: str,
                             aliases: List[str] = None,
                             paper_ids: List[str] = None,
                             dataset_ids: Dict[str, str] = None) -> dict:
        """
        Creates or updates a biomedical node with proper handling of unique identifiers.
        
        Args:
            canonical_name: The primary name for the entity
            node_type: Type of node (disease/gene/genotype/phenotype/treatment)
            aliases: List of alternative names/symbols
            paper_ids: List of paper identifiers that mention this entity
            dataset_ids: Dictionary of {dataset_name: id} for various databases
        
        Returns:
            Dictionary containing the created/updated node properties
        """
        if node_type.lower() not in self.VALID_TYPES:
            raise ValueError(f"Invalid node type. Must be one of: {', '.join(self.VALID_TYPES)}")
        
        # Prepare the query
        query = """
        MERGE (n:Biomedical {canonical_name: $canonical_name, type: $node_type})
        SET n.aliases = $aliases,
            n.paper_ids = $paper_ids,
            n.dataset_ids = $dataset_ids,
            n.last_updated = datetime()
        RETURN n
        """
        
        # Prepare parameters
        params = {
            "canonical_name": canonical_name,
            "node_type": node_type.lower(),
            "aliases": list(set(aliases)) if aliases else [],
            "paper_ids": list(set(paper_ids)) if paper_ids else [],
            "dataset_ids": dataset_ids or {}
        }
        
        with self.get_session() as session:
            result = session.run(query, params)
            return result.single()["n"]

    def add_relationship(self,
                        from_name: str,
                        to_name: str,
                        relationship_type: str,
                        properties: Dict = None) -> dict:
        """
        Creates a relationship between two nodes based on their canonical names.
        
        Args:
            from_name: Canonical name of the source node
            to_name: Canonical name of the target node
            relationship_type: Type of relationship
            properties: Optional dictionary of relationship properties
        """
        query = """
        MATCH (a:Biomedical {canonical_name: $from_name})
        MATCH (b:Biomedical {canonical_name: $to_name})
        MERGE (a)-[r:`$rel_type`]->(b)
        SET r += $properties
        RETURN r
        """
        
        with self.get_session() as session:
            result = session.run(
                query, 
                from_name=from_name,
                to_name=to_name,
                rel_type=relationship_type,
                properties=properties or {}
            )
            return result.single()["r"]

    def update_node_attributes(self,
                             canonical_name: str,
                             new_aliases: List[str] = None,
                             new_paper_ids: List[str] = None,
                             new_dataset_ids: Dict[str, str] = None) -> dict:
        """
        Updates node attributes while maintaining uniqueness of aliases and paper IDs.
        
        Args:
            canonical_name: The primary name of the node to update
            new_aliases: New aliases to add
            new_paper_ids: New paper IDs to add
            new_dataset_ids: New dataset IDs to add/update
        """
        query = """
        MATCH (n:Biomedical {canonical_name: $canonical_name})
        SET n.aliases = $aliases,
            n.paper_ids = $paper_ids,
            n.dataset_ids = $dataset_ids,
            n.last_updated = datetime()
        RETURN n
        """
        
        with self.get_session() as session:
            # First get existing data
            result = session.run(
                "MATCH (n:Biomedical {canonical_name: $name}) RETURN n",
                name=canonical_name
            )
            node = result.single()["n"]
            
            # Merge existing and new data
            current_aliases = set(node.get("aliases", []))
            current_paper_ids = set(node.get("paper_ids", []))
            current_dataset_ids = node.get("dataset_ids", {})
            
            if new_aliases:
                current_aliases.update(new_aliases)
            if new_paper_ids:
                current_paper_ids.update(new_paper_ids)
            if new_dataset_ids:
                current_dataset_ids.update(new_dataset_ids)
            
            # Update the node
            result = session.run(
                query,
                canonical_name=canonical_name,
                aliases=list(current_aliases),
                paper_ids=list(current_paper_ids),
                dataset_ids=current_dataset_ids
            )
            return result.single()["n"]
            
    def verify_database(self) -> bool:
        """Verifies database connection and prints basic information"""
        if not self.driver:
            self.connect()
            
        try:
            # Check available databases
            records, _, _ = self.driver.execute_query("SHOW DATABASES")
            unique_dbs = {record['name']: record.get('home', False) for record in records}
            print("Available databases:")
            for db_name, is_home in unique_dbs.items():
                print(f"- {db_name} {'(home)' if is_home else ''}")
                
            # Test basic query capability
            print("\nTesting connection...")
            with self.get_session() as session:
                # Try to count all nodes (a basic operation that should always work)
                result = session.run("MATCH (n) RETURN count(n) as count")
                count = result.single()["count"]
                print(f"Database contains {count} nodes")
                
                # Get database information (handling multiple components)
                result = session.run("CALL dbms.components() YIELD name, versions, edition")
                components = list(result)
                
                print("\nDatabase Components:")
                for component in components:
                    print(f"- {component['name']} {component['edition']}")
                    print(f"  Version: {component['versions'][0]}")
                
                # Get schema information
                print("\nDatabase Schema:")
                try:
                    # Get node labels and properties
                    result = session.run("""
                        MATCH (n:Biomedical)
                        WITH DISTINCT n.type as type, count(n) as count
                        RETURN collect({type: type, count: count}) as node_types
                    """)
                    node_types = result.single()["node_types"]
                    if node_types:
                        print("\nNode Types:")
                        for type_info in node_types:
                            print(f"- {type_info['type']}: {type_info['count']} nodes")
                    
                    # Get relationship types
                    result = session.run("""
                        MATCH ()-[r]->()
                        WITH DISTINCT type(r) as rel_type, count(r) as count
                        RETURN collect({type: rel_type, count: count}) as rel_types
                    """)
                    rel_types = result.single()["rel_types"]
                    if rel_types:
                        print("\nRelationship Types:")
                        for rel_info in rel_types:
                            print(f"- {rel_info['type']}: {rel_info['count']} relationships")
                    
                except Exception as e:
                    print(f"Note: Could not fetch complete schema: {str(e)}")
            
            return True
        except Exception as e:
            print(f"Verification failed: {str(e)}")
            return False

# Database configuration
URI = "neo4j+s://98e154d4.databases.neo4j.io"
AUTH = (, )
DATABASE =

# Initialize connection
neo4j_db = Neo4jConnection(URI, AUTH, DATABASE)

# Verify connection and print info
neo4j_db.verify_database()

Available databases:
- 98e154d4 (home)
- system 

Testing connection...
Database contains 0 nodes

Database Components:
- Neo4j Kernel enterprise
  Version: 5.27-aura
- Cypher 
  Version: 5

Database Schema:


True

# Neo4j Database Connection Setup

This notebook demonstrates how to properly connect to Neo4j with:
- Connection pooling
- Error handling
- Proper resource cleanup
- Connection verification
- Database selection

In [12]:
from neo4j import GraphDatabase

URI = "neo4j+s://98e154d4.databases.neo4j.io"
AUTH = ("98e154d4", "eiJjs_a56SghJk0HcmaGJgLemDIUy0-45CO_rOki9T4")

driver = GraphDatabase.driver(URI, auth=AUTH)
session = driver.session(database="98e154d4")

    
    

In [28]:
summary = driver.execute_query("""
    CREATE (a:Person {name: $name})
    CREATE (b:Person {name: $friendName})
    CREATE (a)-[:KNOWS]->(b)
    """,
    name="Alice", friendName="David",
    database_="98e154d4",
).summary
print("Created {nodes_created} nodes in {time} ms.".format(
    nodes_created=summary.counters.nodes_created,
    time=summary.result_available_after
))

Created 2 nodes in 2 ms.


In [25]:
records, summary, keys = driver.execute_query("""
    MATCH (p:Person)-[:KNOWS]->(:Person)
    RETURN p.name AS name
    """,
    database_="98e154d4",
)

# Loop through results and do something with them
for record in records:
    print(record.data())  # obtain record as dict

# # Summary information
# print("The query `{query}` returned {records_count} records in {time} ms.".format(
#     query=summary.query, records_count=len(records),
#     time=summary.result_available_after
# ))

{'name': 'Alice'}


In [26]:
driver.execute_query("CALL db.schema.visualization()", database_="98e154d4")

EagerResult(records=[<Record nodes=[<Node element_id='-106' labels=frozenset({'Person'}) properties={'name': 'Person', 'indexes': [], 'constraints': []}>] relationships=[<Relationship element_id='-106' nodes=(<Node element_id='-106' labels=frozenset({'Person'}) properties={'name': 'Person', 'indexes': [], 'constraints': []}>, <Node element_id='-106' labels=frozenset({'Person'}) properties={'name': 'Person', 'indexes': [], 'constraints': []}>) type='KNOWS' properties={'name': 'KNOWS'}>]>], summary=<neo4j._work.summary.ResultSummary object at 0x7f86e489bcd0>, keys=['nodes', 'relationships'])

In [29]:
driver.execute_query("MATCH (n) DETACH DELETE n", database_="98e154d4")

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x7f86e4891650>, keys=[])

In [30]:
session.close()
driver.close()


In [ ]:
# Example: Creating and querying data
try:
    # Create some data
    with neo4j_db.get_session() as session:
        # Create nodes and relationship
        result = session.run("""
            MERGE (a:Person {name: $name})
            MERGE (b:Person {name: $friend_name})
            MERGE (a)-[:KNOWS]->(b)
            RETURN a.name as name, b.name as friend
        """, name="Alice", friend_name="Bob")
        
        # Print results
        for record in result:
            print(f"Created relationship: {record['name']} knows {record['friend']}")
            
    # Query the data
    with neo4j_db.get_session() as session:
        # Find all relationships
        result = session.run("""
            MATCH (p1:Person)-[:KNOWS]->(p2:Person)
            RETURN p1.name as person, p2.name as knows
        """)
        
        # Print results
        print("\nExisting relationships:")
        for record in result:
            print(f"{record['person']} knows {record['knows']}")
            
except Exception as e:
    print(f"Error occurred: {str(e)}")
finally:
    # Always clean up
    neo4j_db.close()

In [ ]:
# Test basic operations
try:
    with neo4j_db.get_session() as session:
        # Create a test node
        print("Creating test node...")
        result = session.run("""
            CREATE (n:Test {name: 'test'})
            RETURN n.name as name
        """)
        print(f"Created node with name: {result.single()['name']}")
        
        # Count nodes of type Test
        result = session.run("""
            MATCH (n:Test)
            RETURN count(n) as count
        """)
        print(f"Number of test nodes: {result.single()['count']}")
        
        # Clean up - remove test nodes
        session.run("MATCH (n:Test) DELETE n")
        print("Test nodes cleaned up")
        
except Exception as e:
    print(f"Error during test: {str(e)}")
    
# Close connection
neo4j_db.close()

In [ ]:
# Example of handling duplicates with merge_data
try:
    # Example data
    genes = [
        {"name": "Gene1", "id": "12345", "score": 0.8},
        {"name": "Gene1", "id": "12345", "score": 0.9},  # Duplicate ID
        {"name": "Gene2", "id": "67890", "score": 0.7}
    ]
    
    print("Adding genes with duplicate handling...")
    for gene in genes:
        # Merge based on ID (will update if exists, create if doesn't)
        node = neo4j_db.merge_data("Gene", gene, unique_fields=["id"])
        print(f"Processed gene: {gene['name']} (ID: {gene['id']})")
    
    # Query to see what was actually created
    with neo4j_db.get_session() as session:
        result = session.run("""
            MATCH (g:Gene)
            RETURN g.name as name, g.id as id, g.score as score
            ORDER BY g.id
        """)
        
        print("\nStored genes:")
        for record in result:
            print(f"Gene: {record['name']}, ID: {record['id']}, Score: {record['score']}")
            
except Exception as e:
    print(f"Error occurred: {str(e)}")
finally:
    # Clean up test data
    with neo4j_db.get_session() as session:
        session.run("MATCH (g:Gene) DELETE g")
        print("\nTest data cleaned up")

# Biomedical Graph Structure

This example demonstrates how to create and manage a biomedical knowledge graph with:
- Node types: disease, gene, genotype, phenotype, treatment
- Properties:
  - canonical_name: Primary identifier
  - aliases: Alternative names/symbols (unique)
  - paper_ids: References to papers
  - dataset_ids: IDs from various databases (unique)
- Automatic handling of duplicates
- Relationship management

In [38]:
# Example: Creating and linking biomedical entities
try:
    # Create a disease node
    disease = neo4j_db.create_biomedical_node(
        canonical_name="Cystic Fibrosis",
        node_type="disease",
        aliases=["CF", "Mucoviscidosis"],
        paper_ids=["PMID:123456", "PMID:789012"],
        dataset_ids={
            "OMIM": "219700",
            "MONDO": "MONDO:0009061"
        }
    )
    print("Created disease node:", disease)

    # Create a gene node
    gene = neo4j_db.create_biomedical_node(
        canonical_name="CFTR",
        node_type="gene",
        aliases=["ABCC7", "CF", "MRP7"],
        paper_ids=["PMID:123456"],
        dataset_ids={
            "HGNC": "1884",
            "ENSEMBL": "ENSG00000001626"
        }
    )
    print("\nCreated gene node:", gene)

    # Create a relationship between disease and gene
    relationship = neo4j_db.add_relationship(
        from_name="CFTR",
        to_name="Cystic Fibrosis",
        relationship_type="ASSOCIATED_WITH",
        properties={
            "confidence": 0.95,
            "evidence": ["PMID:123456"]
        }
    )
    print("\nCreated relationship:", relationship)

    # Update node with new information
    updated_disease = neo4j_db.update_node_attributes(
        canonical_name="Cystic Fibrosis",
        new_aliases=["Fibrocystic Disease of Pancreas"],
        new_paper_ids=["PMID:345678"],
        new_dataset_ids={"SNOMED": "190905008"}
    )
    print("\nUpdated disease node:", updated_disease)

except Exception as e:
    print(f"Error occurred: {str(e)}")
finally:
    # Clean up test data
    with neo4j_db.get_session() as session:
        session.run("MATCH (n:Biomedical) DETACH DELETE n")
        print("\nTest data cleaned up")

Error occurred: {code: Neo.ClientError.Statement.TypeError} {message: Property values can only be of primitive types or arrays thereof. Encountered: Map{MONDO -> String("MONDO:0009061"), OMIM -> String("219700")}.}

Test data cleaned up
